# Week 3 — 02: Feature Engineering

Build all causal features on top of the loaded transaction DataFrame.
Reads `checkpoints/01_*.pkl` and writes `checkpoints/02_df_features.pkl`.

Feature families:
1. Base transaction / local anomaly
2. Causal account history
3. Vectorised causal rolling means & counts
4. Novelty / first-seen
5. Reciprocal / reverse-transaction
6. Behavioural acceleration
7. Network / fan-in / fan-out
8. Corridor / graph
9. TRUE Pattern TXT membership (metadata, not model features)
10. V3 veteran-account anomaly
11. Transaction-derived structural pattern scores


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import state
import numpy as np

df          = state.load("01_df")
pattern_map = state.load("01_pattern_map")
train_idx, val_idx, test_idx = state.load("01_indices")
print(f"Loaded: {len(df):,} rows")


Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_df.pkl
Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_pattern_map.pkl
Loaded checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/01_indices.pkl
Loaded: 5,078,345 rows


In [2]:
from src.features.base      import build_base_features
from src.features.history   import build_history_features
from src.features.rolling   import build_rolling_features
from src.features.novelty   import build_novelty_features
from src.features.reciprocal import build_reciprocal_features
from src.features.acceleration import build_acceleration_features
from src.features.network   import build_network_features
from src.features.corridor  import build_corridor_features
from src.features.pattern_meta import add_pattern_meta
from src.features.veteran   import build_veteran_features
from src.features.structural_scores import build_structural_scores

print("Building base features...")
build_base_features(df)
print(f"  columns: {len(df.columns)}")

print("Building history features...")
build_history_features(df)
print(f"  columns: {len(df.columns)}")


Building base features...
  columns: 36
Building history features...
  columns: 50


In [3]:
print("Building rolling features (slow ~5 min)...")
build_rolling_features(df)
print(f"  columns: {len(df.columns)}")


Building rolling features (slow ~5 min)...


Rolling amount families:   0%|          | 0/4 [00:00<?, ?family/s]

sender_paid: rolling windows:   0%|          | 0/3 [00:00<?, ?window/s]

sender_received: rolling windows:   0%|          | 0/3 [00:00<?, ?window/s]

receiver_paid: rolling windows:   0%|          | 0/3 [00:00<?, ?window/s]

...

sender_activity: activity windows:   0%|          | 0/2 [00:00<?, ?window/s]

receiver_activity: activity windows:   0%|          | 0/2 [00:00<?, ?window/s]

pair_activity: activity windows:   0%|          | 0/2 [00:00<?, ?window/s]

Current-vs-history ratios:   0%|          | 0/6 [00:00<?, ?feature/s]

  columns: 74


In [4]:
print("Building novelty features...")
build_novelty_features(df)

print("Building reciprocal features...")
build_reciprocal_features(df)

print("Building acceleration features...")
build_acceleration_features(df)

print("Building network features...")
build_network_features(df)

print("Building corridor features...")
build_corridor_features(df)
print(f"  columns: {len(df.columns)}")


Building novelty features...
Building reciprocal features...
Building acceleration features...


Behaviour acceleration:   0%|          | 0/7 [00:00<?, ?feature/s]

Building network features...


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/network.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fanin_burst_20"]  = df.receiver_activity_20.astype("float32")
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/network.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fanout_burst_20"] = df.sender_activity_20.astype("float32")
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-I

Building corridor features...


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/helpers.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame[name] = (
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/corridor.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["corridor_share_sender"]    = safe_ratio(df.corridor_prev_count.to_numpy(), df.sender_prev_count.to_numpy()   + 1)
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/

  columns: 123


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/corridor.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["corridor_share_x_anomaly"] = (df.corridor_share_sender * df.corridor_established_anomaly).astype("float32")


In [5]:
print("Adding pattern metadata...")
df = add_pattern_meta(df, pattern_map)

print("Building V3 veteran features...")
build_veteran_features(df)

print("Building structural scores...")
build_structural_scores(df, train_idx)
print(f"  columns: {len(df.columns)}")

state.save("02_df_features", df)
print("Checkpoint 02 saved.")


Adding pattern metadata...


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/pattern_meta.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["txt_patterned"]            = df["txt_pattern_memberships"].gt(0).astype("int8")
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/veteran.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["sender_veteran_flag"]  = veteran.astype("int8")
/home/malnaser/EN_Workspace/DEVELOPMENT/_play

Building V3 veteran features...


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/veteran.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["sender_diversity_per_history"]  = ((su + 1.0) / (sender_prev + 1.0)).clip(0, 1).astype("float32")
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/veteran.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["sender_diversity_x_veteran"]    = (su * veteran).astype("float32")
/home/malnase

Building structural scores...


/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/structural_scores.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = np.clip(np.nan_to_num(values, nan=0, posinf=1, neginf=0), 0, 1).astype("float32")
/home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/src/features/structural_scores.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = np.clip(np.nan_to_num(values, nan=0, posinf=1, neginf=0), 0, 1

  columns: 156
Saved checkpoint: /home/malnaser/EN_Workspace/DEVELOPMENT/_playground/DS-Internship-Summer-2026/aml-detection-project/checkpoints/02_df_features.pkl
Checkpoint 02 saved.
